In [1]:
import json

with open("tree_demand.json", "r", encoding="utf-8") as f:
    data = json.load(f)
trees={}
for tree in data:
    page_source = tree["page_source"]
    trees[page_source] = tree["content"]

In [2]:
with open("re.json", "r", encoding="utf-8") as file_0:
    demand = json.load(file_0)
contexts = {}
purposes={}
for data in demand:
    page_source = data['page_source']
    purpose = data['purpose']
    datas = []
    for inputs in data['data']:
        datas.append(inputs)
    contexts[page_source]=datas
    purposes[page_source]=purpose

In [7]:
def generate_demand(info, tree):
    result = ""
    for key, value in info.items():
        for item in tree:
            if key == item['key']:
                result = result + str(key) + "(" + item['annotation'] + ")" + ":" + str(value) + "\n"
    return result


import requests
from urllib.parse import quote


def gpt(message, tree, purpose):
    # 对 user_msg 进行 URL 编码
    prompt = ''''
    首先你对需求列表中的键值进行一些联想（think step by step），将需求列表中具体的值改写成描述性的话或者同义词，然后再回答客服的问题。回答语气不要太正式，需求描述应该符合中文口语习惯。
    ##
    目的：租房子
    客服问题:您好，请问您对想要租住的房屋的房间数量有什么要求呢？
    需求列表:
    户型（房屋的房间数量或居室规模）:三室
    装修（房屋装饰和装潢的各种程度和类型）:精装修
    think step by step：
    三室的户型房间数量属于中等，因此\'''户型为三室\'''，可以用\'''房间数量中等\'''替换
    精装修的房子也就是装修非常精美，因此\'''装修为精装修\'''，可以用\'''装修精美\'''替换
    user:我觉得房间数量中等就好，不用太多。但是我希望装修可以精美一些。

    ##
    目的:买二手车
    客服问题:您好，请问您对车辆的价格有什么需求吗，或者说车的车型，车的特点等等，如果您有别的需求也请告诉我
    需求列表:
    能源（车辆的能源类型）:纯电动
    think step by step：
    纯电动的车不会排放尾气，比较环保，因此\'''能源为纯电动\'''，可以用\'''环保\'''替换
    user:我希望能找到一辆环保的，对环境友好的车。

    ##
    目的:'''
    re = generate_demand(message['response'], tree)
    user_msg = prompt + purpose + "\n客服问题:您好，请问您有什么需求呢？" + "\n需求列表:\n" + re
    encoded_user_msg = quote(user_msg)

    resp = requests.post(f"http://127.0.0.1:8081/process_messages?question={encoded_user_msg}")
    if resp.status_code == 200:
        return resp.text


class DEMAND:
    def __init__(self):
        self.input = ""
        self.response = {}

from tqdm import tqdm

re_contexts = []
for key, context_data in tqdm(contexts.items()):
    page_source = key
    ami_tree = trees[page_source]
    purpose = purposes[page_source]
    re_contexts = []
    tree_json = []
    for context in tqdm(context_data[9:10]):
        demand = DEMAND()
        re_context = gpt(context,ami_tree,purpose)
        start_index = re_context.find("user:") + len("user:")
        extracted_statement = re_context[start_index:].strip()
        demand.input = extracted_statement
        demand.response = context["response"]
        re_contexts.append(demand)
    for con in re_contexts:
        tree_dict = {
            "input": con.input,
            "response": con.response
        }
        tree_json.append(tree_dict)
    tree_dict = {
        "page_source": page_source,
        "content": tree_json
    }
    with open("re_write.json", "a", encoding="utf-8") as f:
        json.dump(tree_dict, f, indent=4, ensure_ascii=False)

  0%|          | 0/42 [00:00<?, ?it/s]


KeyboardInterrupt: 

In [39]:
import json
with open("re_write.json", "r", encoding="utf-8") as file_0:
    demand = json.load(file_0)
purposes={}
contexts={}
for data in demand:
    page_source = data["page_source"]
    contexts[page_source] = data["content"]

In [114]:
context_to_be_done = contexts["id4_scrnli_2023_8_9 10-56-38.png"]


In [115]:
prompt = '''instruction:input中关于"字数"，"更新时间"之类的描述太过模糊，请你改写它。

input：我想看一本长度适中的，包含奇幻元素的小说。
demand：
字": 30万字以下,
分类: 玄幻
output：我想看一本20万字左右，包含奇幻元素的小说。

input：我想看一部最近更新的中长篇小说，最好是幽默风格的。
demand：
字数: 50-100万字,
分类: 轻小说
更新时间: 半月内,
output：我想看一部15天内更新的60多万字的小说，最好是幽默风格的。

input：我希望找一些免费阅读的、最近有更新的幽默轻松的小说。
demand：
属性: 免费,
更新时间: 三日内,
分类: 轻小说
output：我希望找一些免费阅读的、三日内有更新的幽默轻松的小说。
input:'''

In [116]:
def generate(sen):
    results=""
    for key,value in sen.items():
        results+=str(key)+":"+str(value)+"\n"
    return results

In [117]:
import requests
from urllib.parse import quote
from tqdm import tqdm
datas=[]
for con in tqdm(context_to_be_done) :
    sen = con['input']
    demand = con['response']
    demand_list = generate(demand)
    input_sen = prompt+sen+"\ndemand:\n"+demand_list+"\noutput:"
    encoded_user_msg = quote(input_sen)

    resp = requests.post(f"http://127.0.0.1:8081/process_messages?question={encoded_user_msg}")
    if resp.status_code == 200:

        re_write={
            "input":resp.text,
            "response":demand
        }
        datas.append(re_write)
with open("need_tobe_change.json","w",encoding='utf-8') as file:
    json.dump(datas, file, indent=4, ensure_ascii=False)


100%|██████████| 81/81 [02:16<00:00,  1.68s/it]
